Neither Wilcoxon, nor paired t-test can be used because ...


In [1]:
import os
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
import math
from tqdm import tqdm
from collections import defaultdict
from itertools import product
from typing import Tuple, Optional

In [2]:
sp18_ext_path = "../resources/developed/sp18/external_effect"

res_outdir = "../out/external_effect"

incsv_args = {
	"sep":"\t",
    "header":0,
    "index_col":"const_comb",
    # all str, WC int, is linker pluralic bool
    "dtype": defaultdict(
        lambda: str,
        wc="int",
        is_linker_pl="bool"
    ),
	# = csv.QUOTE_NONE, separate by tabs only
    "quoting": 3
}

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

In [3]:
def calc_pll_shares(
    sp18_sample: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # calculate absolute counts and shares of compounds 
    # with pluralic linkers in Sg vs. Pl forms

    sp18_sample_pll_counts_data = {}
    for const_comb, group in sp18_sample.groupby("const_comb"):
        for num in ["sg", "pl"]:
            sp18_sample_pll_counts_data[(const_comb, f"num_{num}")] = {
                "l_sg": len(
                    group[(group["n2_num"] == num) &
                    (~group["is_linker_pl"])]
                ),
                "l_pl": len(
                    group[(group["n2_num"] == num) &
                    (group["is_linker_pl"])]
                )
            }

    sp18_sample_pllcounts = pd.DataFrame(sp18_sample_pll_counts_data).T
    sp18_sample_pllcounts["pl_share"] = sp18_sample_pllcounts["l_pl"] / (
        sp18_sample_pllcounts["l_sg"] + sp18_sample_pllcounts["l_pl"]
    )

    sp18_sample_pllshares = sp18_sample_pllcounts \
        .unstack()["pl_share"]  \
        [["num_sg", "num_pl"]]

    return sp18_sample_pllcounts, sp18_sample_pllshares

In [ ]:
def sp18_orig_analysis(
    contingency_table: pd.DataFrame,
    label_1: str,
    label_2: str
) -> pd.DataFrame:

    # Methodology from the original [schaefer_pankratz_2018:26] paper:
    # 0. Create contingency table:
    # n2_num (Sg vs. Pl) x n1_pl_marker (non- vs. pluralic linker).
    # 1. Perform the chi-squared test of independence over the table
    # to assess the statistical significance of the association between
    # the Pl form of N2 and the presence of a pluralic linker in N1.
    # 2. Perform the Cramer's V test to assess the strength
    # of the association.
    # 3. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # 1. Perform the chi-squared test.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `chi2_contingency` function from `scipy.stats`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    chi2_res = chi2_contingency(contingency_table)

    # 2. Perform the Cramer's V test to assess 
    # the strength of the association.

    # the original R code for Cramer's V is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   cramer.v <- function(chisq) {
    #       unname(sqrt(chisq$statistic/(sum(chisq$observed)*min(dim(chisq$observed)-1))))
    #   }
    #
    # we will reconstruct it in Python:
    #   dim(chisq$observed) -> ic_alt_nipe_sg_l_ct.shape
    #
    #   min(dim(chisq$observed)-1) -> min(
    #       ic_alt_nipe_sg_l_ct.shape[0] - 1,
    #       ic_alt_nipe_sg_l_ct.shape[1] - 1
    #   )
    # 
    #   sum(chisq$observed) -> ic_alt_nipe_sg_l_ct.values.sum()
    #
    #   unname -> [irrelevant for Python]
    cramer_v_val = math.sqrt(
        chi2_res.statistic / (
            contingency_table.values.sum() *
            min(
                contingency_table.shape[0] - 1,
                contingency_table.shape[1] - 1
            )
        )
    )

    # DEPRECATED: for 2x2 contingency tables, Cohen's W
    # is effectively identical to Cramer's V
    # # the original R code for Cohen's W is available in
    # # linkingelements/Data/R/functions.R, and is as follows:
    # # 
    # #   # cohen.w <- function(chisq) {
    # #       cramer.v(chisq) * sqrt(min(dim(chisq$observed)-1))
    # #   }
    # cohen_w_val = cramer_v_val * math.sqrt(
    #     min(
    #         contingency_table.shape[0] - 1,
    #         contingency_table.shape[1] - 1
    #     )
    # )

    # 3. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `standardized_resids` property of
    # `statsmodels.stats.contingency_tables.Table`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    resids = Table(contingency_table)    \
        .standardized_resids.round(3)
    
    chi2df = pd.Series(
        data={
            "N": contingency_table.values.sum(),
            "df": chi2_res.dof,
            "chi2": chi2_res.statistic.round(3),
            "p": chi2_res.pvalue,
            "cramer_v": round(cramer_v_val, 3),
            # "cohen_w": round(cohen_w_val, 3),
            "resids_cong": resids.loc["l_pl", label_1].item(),
            "resids_incong": resids.loc["l_pl", label_2].item()
        }
    ).round(3)
    chi2df["N"] = chi2df["N"].astype(int)
    chi2df["df"] = chi2df["df"].astype(int)

    return chi2df

In [5]:
keys = [
        # Compounds built from non-unique constituent combinations (regardless 
        # of the linker) appearing both in the Sg and the Pl forms
        # and not undergoing the IntPE whose N1 may not attach a pluralic linker
        # (N=1236, UniqN=665, UniqCN=468, UniqN1N=30)
        "ic_alt_nipe_npll",

        # Compounds built from non-unique constituent combinations (regardless 
        # of the linker) appearing both in the Sg and the Pl forms
        # and not undergoing the IntPE
        # (N=1931, UniqN=1043, UniqCN=724)
        "ic_alt_nipe",

        # Compounds built from non-unique constituent combinations (regardless 
        # of the linker) appearing both in the Sg and the Pl forms
        # (N=2318, UniqN=1246, UniqCN=869)
        "ic_alt"
    ]

freq_levels = [
    ("lfreq", 0), # all frequencies, >= 0
    ("mfreq", 20),  # moderate frequency, >= 20
    ("hfreq", 100)  # high frequency, >= 100
]

# Study 1

Compound groups

In [ ]:
def run_study_1(
	sp18_sample: pd.DataFrame,
	outdir: Optional[str]=None
) -> Tuple[
	pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series
]:

	# calculate absolute counts and shares of compounds 
	# with pluralic linkers in Sg vs. Pl forms
	(
		sp18_sample_pllcounts,
		sp18_sample_pllshares
	) = calc_pll_shares(sp18_sample)

	# shares desc
	sp18_sample_pllshares_desc = sp18_sample_pllshares.describe()

	# create contingency table replicating [schaefer_pankratz_2018:26]
	# n2_num (Sg vs. Pl) x n1_pl_marker (non- vs. pluralic linker).
	contingency_table = sp18_sample_pllcounts \
		.groupby(level=1)[["l_sg", "l_pl"]].sum()   \
		.T  \
		# [["num_sg", "num_pl"]]
	# perform the original analysis from [schaefer_pankratz_2018:26]
	chi2df = sp18_orig_analysis(
		contingency_table,
		label_1="num_pl",
		label_2="num_sg"
	)

	if outdir:

		os.makedirs(outdir, exist_ok=True)

		sp18_sample_pllcounts.to_csv(
			os.path.join(outdir, "pllcounts.tsv"),
			**outcsv_args
		)
		sp18_sample_pllshares.to_csv(
			os.path.join(outdir, "pllshares.tsv"),
			**outcsv_args
		)

		sp18_sample_pllshares_desc.to_csv(
			os.path.join(outdir, "pllshares_desc.tsv"),
			**outcsv_args
		)

		contingency_table.to_csv(
			os.path.join(outdir, "contingency_table.tsv"),
			**outcsv_args
		)
		chi2df.to_csv(
			os.path.join(outdir, "chi2.tsv"),
			**{**outcsv_args, "header": False}
		)

	return (
		sp18_sample_pllcounts, sp18_sample_pllshares, 
		sp18_sample_pllshares_desc,
		contingency_table, chi2df
	)

In [ ]:
chi2_summaries = []
with tqdm(
    total=len(keys) * len(freq_levels), 
    desc="SP18 compound group ExtPE study"
) as pbar:

    for sample_sfx in keys:

        sp18_sample = pd.read_csv(
            os.path.join(sp18_ext_path, f"{sample_sfx}.tsv"),
            **incsv_args
        )

        chi2dfs = []
        outdir = os.path.join(res_outdir, sample_sfx)

        # three levels of frequency
        for freq_sfx, freq_threshold in freq_levels:

            sp18_subsample = sp18_sample[sp18_sample["wc"] >= freq_threshold]
            *_, chi2df = run_study_1(sp18_subsample, os.path.join(outdir, freq_sfx))
            chi2df.name = freq_sfx
            chi2dfs.append(chi2df)
            
            pbar.update(1)

        chi2_summary = pd.concat(chi2dfs, axis=1)
        chi2_summary.to_csv(
            os.path.join(outdir, "chi2_summary.tsv"),
            **outcsv_args
        )
        chi2_summaries.append(chi2_summary)

SP18 compound group ExtPE study: 100%|██████████| 9/9 [00:03<00:00,  2.62it/s]


In [8]:
chi2_summary = pd.concat(chi2_summaries, keys=keys)  \
    .swaplevel(0, 1)    \
    .sort_index(level=0, sort_remaining=False)  \
    .reindex(chi2df.index, level=0)
chi2_summary.to_csv(
    os.path.join(res_outdir, "chi2_summary.tsv"),
    **outcsv_args
)
chi2_summary

lfreq     mfreq     hfreq
N             ic_alt_nipe_npll  1235.000   808.000   694.000
              ic_alt_nipe       1930.000  1248.000  1076.000
              ic_alt            2317.000  1516.000  1308.000
df            ic_alt_nipe_npll     1.000     1.000     1.000
              ic_alt_nipe          1.000     1.000     1.000
              ic_alt               1.000     1.000     1.000
chi2          ic_alt_nipe_npll     1.785     0.049     0.026
              ic_alt_nipe          2.051     0.139     0.000
              ic_alt               1.884     0.175     0.012
p             ic_alt_nipe_npll     0.181     0.824     0.872
              ic_alt_nipe          0.152     0.710     1.000
              ic_alt               0.170     0.676     0.914
cramer_v      ic_alt_nipe_npll     0.038     0.008     0.006
              ic_alt_nipe          0.033     0.011     0.000
              ic_alt               0.029     0.011     0.003
resids_cong   ic_alt_nipe_npll     1.394     0.296    -0.242
              ic_alt_nipe          1.478     0.430     0.028
              ic_alt               1.414     0.470     0.163
resids_incong ic_alt_nipe_npll    -1.394    -0.296     0.242
              ic_alt_nipe         -1.478    -0.430    -0.028
              ic_alt              -1.414    -0.470    -0.163

# Study 2

Individual N1s.

In [9]:
n1s = sp18_sample["n1"].unique()
n1s

<StringArray>
[      'apfel',         'bad',        'ball',        'bett',        'bild',
       'brett',       'brief',        'buch',       'bucht',       'dämon',
          'ei', 'eigenschaft',     'element',        'frau',       'gerät',
    'geräusch',    'geschenk',     'gitarre',        'hand',        'haus',
        'hemd',        'hund',       'katze',        'kind',     'kommune',
        'lied',        'loch',      'mutter',       'nagel',      'nation',
         'ohr',      'person',     'produkt',   'projektor',         'rad',
     'schloss',     'schwert',   'schwester',       'sonne',       'stadt',
       'vater',       'vogel',         'weg',        'wurm',        'zahn',
       'zitat']
Length: 46, dtype: str

In [10]:
chi2_summaries = defaultdict(dict)
sign_n1s = []
with tqdm(
    total=len(keys) * len(freq_levels) * len(n1s), 
    desc="SP18 N1 ExtPE study"
) as pbar:

    for sample_sfx, n1 in product(keys, n1s):

        sp18_sample = pd.read_csv(
            os.path.join(sp18_ext_path, f"{sample_sfx}.tsv"),
            **incsv_args
        )

        chi2dfs = []
        is_sign = False

        # three levels of frequency
        for freq_sfx, freq_threshold in freq_levels:

            sp18_subsample = sp18_sample[
                (sp18_sample["wc"] >= freq_threshold) &
                (sp18_sample["n1"] == n1)
            ]

            try:
                *_, chi2df = run_study_1(sp18_subsample)
                chi2df.name = freq_sfx
                chi2dfs.append(chi2df)
                # increased p-value to account for small sample sizes;
                # the final decision will be based on Cramer's V and resids
                if chi2df["p"] < 0.1:
                    sign_n1s.append(n1)
            except KeyError, ValueError:
                # all_nipe_npll is missing a few N1s;
                # otherwise, after freq filtering
                # not all CCs have valid contingency tables
                pass

            pbar.update(1)
            
        chi2_summary = pd.concat(chi2dfs, axis=1) if chi2dfs else None
        chi2_summaries[n1][sample_sfx] = chi2_summary

SP18 N1 ExtPE study: 100%|██████████| 414/414 [00:05<00:00, 71.86it/s] 


In [11]:
for n1, chi2_sum in chi2_summaries.items():

    chi2_summary = pd.concat(chi2_sum.values(), keys=chi2_sum.keys())  \
        .swaplevel(0, 1)    \
        .sort_index(level=0, sort_remaining=False)  \
        .reindex(chi2df.index, level=0)
    chi2_summary.to_csv(
        os.path.join(res_outdir, "n1", "all", f"{n1}.tsv"),
        **outcsv_args
    )

In [26]:
# TODO: optimize
# rerun significant N1s and save the outputs
chi2_summaries = defaultdict(dict)
with tqdm(
    total=len(keys) * len(freq_levels) * len(sign_n1s), 
    desc="SP18 significant N1 ExtPE study"
) as pbar:
    
    for sample_sfx, n1 in product(keys, sign_n1s):

        sp18_sample = pd.read_csv(
            os.path.join(sp18_ext_path, f"{sample_sfx}.tsv"),
            **incsv_args
        )

        chi2dfs = []
        outdir = os.path.join(res_outdir, "n1", n1, sample_sfx)
        is_sign = False

        # three levels of frequency
        for freq_sfx, freq_threshold in freq_levels:

            sp18_subsample = sp18_sample[
                (sp18_sample["wc"] >= freq_threshold) &
                (sp18_sample["n1"] == n1)
            ]

            try:
                *_, chi2df = run_study_1(
                    sp18_subsample,
                    os.path.join(outdir, freq_sfx)
                )
                chi2df.name = freq_sfx
                chi2dfs.append(chi2df)
            except KeyError, ValueError:
                # all_nipe_npll is missing a few N1s;
                # otherwise, after freq filtering
                # not all CCs have valid contingency tables
                pass

            pbar.update(1)
            
        chi2_summary = pd.concat(chi2dfs, axis=1) if chi2dfs else None
        if chi2_summary is not None:
            chi2_summary.to_csv(
                os.path.join(outdir, "chi2_summary.tsv"),
                **outcsv_args
            )
            chi2_summaries[n1][sample_sfx] = chi2_summary

SP18 significant N1 ExtPE study: 100%|██████████| 63/63 [00:01<00:00, 62.92it/s]


In [27]:
for n1, chi2_sum in chi2_summaries.items():

    chi2_summary = pd.concat(chi2_sum.values(), keys=chi2_sum.keys())  \
        .swaplevel(0, 1)    \
        .sort_index(level=0, sort_remaining=False)  \
        .reindex(chi2df.index, level=0)
    chi2_summary.to_csv(
        os.path.join(res_outdir, "n1", n1, "chi2_summary.tsv"),
        **outcsv_args
    )

In [14]:
apfel = pd.concat(chi2_summaries["apfel"])  \
    .swaplevel(0, 1)    \
    .sort_index(level=0, sort_remaining=False)  \
    .reindex(chi2df.index, level=0)

In [15]:
apfel.loc[["p", "cramer_v"]]

lfreq  mfreq  hfreq
p        ic_alt_nipe_npll  0.072  0.712  0.276
         ic_alt_nipe       0.072  0.712  0.276
         ic_alt            0.135  0.712  0.276
cramer_v ic_alt_nipe_npll  0.265  0.064  0.214
         ic_alt_nipe       0.265  0.064  0.214
         ic_alt            0.207  0.064  0.214

In [16]:
person = pd.concat(chi2_summaries["person"])  \
    .swaplevel(0, 1)    \
    .sort_index(level=0, sort_remaining=False)  \
    .reindex(chi2df.index, level=0)

In [17]:
person.loc[["p", "cramer_v"]]

lfreq
p        ic_alt_nipe_npll  0.052
         ic_alt_nipe       0.052
         ic_alt            0.046
cramer_v ic_alt_nipe_npll  0.272
         ic_alt_nipe       0.272
         ic_alt            0.245

# Study 3

Frequency effects.

In [ ]:
def run_study_3(
	sp18_sample_1: pd.DataFrame,
	sp18_sample_2: pd.DataFrame,
	label_1: str,
	label_2: str,
	outdir: Optional[str]=None
) -> Tuple[
	pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series
]:

	# calculate absolute counts of compounds 
	# with pluralic linkers in Pl forms
	# in the two settings
	pllcounts_1, pllshares_1 = calc_pll_shares(sp18_sample_1)
	pllcounts_2, pllshares_2 = calc_pll_shares(sp18_sample_2)
	# extract only values in Pl form
	pllcounts_1 = pllcounts_1.swaplevel(0, 1).loc["num_pl"]
	pllcounts_2 = pllcounts_2.swaplevel(0, 1).loc["num_pl"]
	# since we have different portions of data,
	# some constituent combinations will be
	# non-existent in one of the samples; force
	# mask them with NaN before joining
	all_ccs = pllcounts_1.index.union(pllcounts_2.index)
	pllcounts_1 = pllcounts_1.reindex(all_ccs)
	pllcounts_2 = pllcounts_2.reindex(all_ccs)
	sp18_samples_pllcounts = pd.concat(
		(pllcounts_1, pllcounts_2),
		keys=[label_1, label_2]
	).swaplevel(0, 1)	\
	.sort_index(level=0, sort_remaining=False)
		
	# shares desc
	pllshares_1 = pllshares_1["num_pl"]
	pllshares_1.name = label_1
	pllshares_2 = pllshares_2["num_pl"]
	pllshares_2.name = label_2
	sp18_samples_pllshares = pd.concat((pllshares_1, pllshares_2), axis=1)
	sp18_samples_pllshares_desc = sp18_samples_pllshares.describe()

	contingency_table = sp18_samples_pllcounts \
		.groupby(level=1).sum()   \
		.T

	chi2df = sp18_orig_analysis(
		contingency_table,
		label_1=label_1,
		label_2=label_2
	)

	if outdir:

		os.makedirs(outdir, exist_ok=True)

		sp18_samples_pllcounts.to_csv(
			os.path.join(outdir, "pllcounts.tsv"),
			**outcsv_args
		)
		sp18_samples_pllshares.to_csv(
			os.path.join(outdir, "pllshares.tsv"),
			**outcsv_args
		)

		sp18_samples_pllshares_desc.to_csv(
			os.path.join(outdir, "pllshares_desc.tsv"),
			**outcsv_args
		)

		contingency_table.to_csv(
			os.path.join(outdir, "contingency_table.tsv"),
			**outcsv_args
		)
		chi2df.to_csv(
			os.path.join(outdir, "chi2.tsv"),
			**{**outcsv_args, "header": False}
		)

	return (
		sp18_samples_pllcounts, sp18_samples_pllshares, 
		sp18_samples_pllshares_desc,
		contingency_table, chi2df
	)

In [32]:
# problematic groups
sp18_sample_2 = pd.read_csv(
    os.path.join(sp18_ext_path, f"{keys[0]}.tsv"),
    **incsv_args
)
with tqdm(total=len(keys) - 1, desc="SP18 data quality ExtPE study") as pbar:

    chi2dfs = []

    for sample_sfx in keys[1:]:

        sp18_sample_1 = pd.read_csv(
            os.path.join(sp18_ext_path, f"{sample_sfx}.tsv"),
            **incsv_args
        )

        outdir = os.path.join(res_outdir, "paired", "data_quality")

        # we expect more pluralic linkers in the first sample
        *_, chi2df = run_study_3(
            sp18_sample_1,
            sp18_sample_2,
            label_1=sample_sfx,
            label_2=keys[0],
            outdir=os.path.join(outdir, f"{keys[0]}_vs_{sample_sfx}")
        )
        chi2df.name = f"{keys[0]}_vs_{sample_sfx}"
        chi2dfs.append(chi2df)
        
        pbar.update(1)

    chi2_summary = pd.concat(chi2dfs, axis=1)
    chi2_summary.to_csv(
        os.path.join(outdir, "chi2_summary.tsv"),
        **outcsv_args
    )
    chi2_summaries.append(chi2_summary)

SP18 data quality ExtPE study: 100%|██████████| 2/2 [00:01<00:00,  1.02it/s]


In [33]:
chi2_summary

,ic_alt_nipe_npll_vs_ic_alt_nipe,ic_alt_nipe_npll_vs_ic_alt
N,1560.000,1735.000
df,1.000,1.000
chi2,0.275,7.635
p,0.600,0.006
cramer_v,0.013,0.066
resids_cong,0.577,2.814
resids_incong,-0.577,-2.814


In [34]:
# frequency groups
with tqdm(
    total=len(keys) * (len(freq_levels) - 1),
    desc="SP18 data quality ExtPE study"
) as pbar:

    chi2_summaries = []

    for sample_sfx in keys:

        sp18_sample_2 = pd.read_csv(
            os.path.join(sp18_ext_path, f"{sample_sfx}.tsv"),
            **incsv_args
        )

        chi2dfs = []
        outdir = os.path.join(res_outdir, "paired", "freq", sample_sfx)

        # three levels of frequency
        for freq_sfx, freq_threshold in freq_levels[1:]:

            sp18_sample_1 = sp18_sample_2[(sp18_sample_2["wc"] >= freq_threshold)]

            try:
                # we expect more pluralic linkers in the first sample
                *_, chi2df = run_study_3(
                    sp18_sample_1,
                    sp18_sample_2,
                    label_1=freq_sfx,
                    label_2="lfreq",
                    outdir=os.path.join(outdir, f"lfreq_vs_{freq_sfx}")
                )
                chi2df.name = f"lfreq_vs_{freq_sfx}"
                chi2dfs.append(chi2df)
            except KeyError, ValueError:
                # all_nipe_npll is missing a few N1s;
                # otherwise, after freq filtering
                # not all CCs have valid contingency tables
                pass

            pbar.update(1)

        chi2_summary = pd.concat(chi2dfs, axis=1) if chi2dfs else None
        if chi2_summary is not None:
            chi2_summary.to_csv(
                os.path.join(outdir, "chi2_summary.tsv"),
                **outcsv_args
            )
            chi2_summaries.append(chi2_summary)

SP18 data quality ExtPE study: 100%|██████████| 6/6 [00:05<00:00,  1.19it/s]


In [36]:
chi2_summary = pd.concat(chi2_summaries, keys=keys)  \
    .swaplevel(0, 1)    \
    .sort_index(level=0, sort_remaining=False)  \
    .reindex(chi2df.index, level=0)
chi2_summary.to_csv(
    os.path.join(res_outdir, "paired", "freq", "chi2_summary.tsv"),
    **outcsv_args
)
chi2_summary

lfreq_vs_mfreq  lfreq_vs_hfreq
N             ic_alt_nipe_npll        1010.000         953.000
              ic_alt_nipe             1580.000        1494.000
              ic_alt                  1876.000        1772.000
df            ic_alt_nipe_npll           1.000           1.000
              ic_alt_nipe                1.000           1.000
              ic_alt                     1.000           1.000
chi2          ic_alt_nipe_npll           4.376           7.755
              ic_alt_nipe                1.718           2.933
              ic_alt                     0.939           1.952
p             ic_alt_nipe_npll           0.036           0.005
              ic_alt_nipe                0.190           0.087
              ic_alt                     0.332           0.162
cramer_v      ic_alt_nipe_npll           0.066           0.090
              ic_alt_nipe                0.033           0.044
              ic_alt                     0.022           0.033
resids_cong   ic_alt_nipe_npll          -2.158          -2.854
              ic_alt_nipe               -1.363          -1.767
              ic_alt                    -1.016          -1.446
resids_incong ic_alt_nipe_npll           2.158           2.854
              ic_alt_nipe                1.363           1.767
              ic_alt                     1.016           1.446